# ⚡ Notebook 08: ASSUME 仿真结果分析

## 学习目标
1. 读取 ASSUME 仿真输出文件 (CSV)
2. 可视化出清价格、发电调度和智能体利润
3. 对比不同场景 (基准 / 高风电 / 夏季高峰)
4. 理解市场出清机制和供需平衡

## 数据来源

本 notebook 使用 `run_simulation.py` 生成的输出文件。
建议先运行三个场景:
```bash
python assume/run_simulation.py --config assume/configs/assume_china_config.yaml --output outputs/base
python assume/run_simulation.py --config assume/configs/assume_china_wind_high.yaml --output outputs/wind_high
python assume/run_simulation.py --config assume/configs/assume_china_summer_peak.yaml --output outputs/summer_peak
```

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from pathlib import Path

pio.renderers.default = 'notebook'

---
## 步骤 1: 加载仿真结果

定义辅助函数加载指定场景的 4 个输出文件。

In [ ]:
def load_simulation_results(scenario_dir):
    """Load all output files from a simulation run.
    
    Returns:
        dict with keys: prices, dispatch, profits, metadata
    """
    d = Path(scenario_dir)
    
    prices = pd.read_csv(d / 'clearing_prices.csv', parse_dates=['timestamp'])
    dispatch = pd.read_csv(d / 'dispatch.csv', parse_dates=['timestamp'])
    profits = pd.read_csv(d / 'agent_profits.csv', parse_dates=['timestamp'])
    
    import json
    with open(d / 'simulation_metadata.json') as f:
        metadata = json.load(f)
    
    return {'prices': prices, 'dispatch': dispatch, 'profits': profits, 'metadata': metadata}


# ── 加载各个场景 ──
scenarios = {}
for name in ['base', 'wind_high', 'summer_peak']:
    path = f'../outputs/{name}'
    try:
        scenarios[name] = load_simulation_results(path)
        print(f"✓ {name}: loaded ({len(scenarios[name]['prices'])} hours)")
    except FileNotFoundError:
        print(f"✗ {name}: not found at {path}")

---
## 步骤 2: 出清价格分析

绘制各场景的出清价格时序。

**关键概念**:
- 出清价格 = 最后一个中标机组的边际成本 (pay-as-clear)
- 价格上限 1500 元/MWh (中国省间现货规则)
- 夏季高峰场景需求增加 → 更多高价机组 (气电) 中标 → 价格上升

In [ ]:
colors = {'base': '#1f77b4', 'wind_high': '#2ca02c', 'summer_peak': '#d62728'}

fig = go.Figure()

for name, data in scenarios.items():
    df = data['prices']
    fig.add_trace(go.Scatter(
        x=df['timestamp'],
        y=df['price_da_元_per_mwh'],
        mode='lines',
        name=name,
        line=dict(width=2, color=colors.get(name, '#999')),
    ))

fig.add_hline(y=1500, line_dash='dash', line_color='red',
              annotation_text='Price Cap (1500 元/MWh)')
fig.add_hline(y=0, line_dash='dash', line_color='gray',
              annotation_text='Price Floor (0 元/MWh)')

fig.update_layout(
    title='Clearing Prices by Scenario — 7-Day Simulation',
    xaxis_title='Time',
    yaxis_title='Price (元/MWh)',
    hovermode='x unified',
    height=500,
    legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01),
)
fig.show()

### 📊 价格统计对比

In [ ]:
price_stats = []
for name, data in scenarios.items():
    p = data['prices']['price_da_元_per_mwh']
    price_stats.append({
        'Scenario': name,
        'Mean': f"{p.mean():.0f}",
        'Min': f"{p.min():.0f}",
        'Max': f"{p.max():.0f}",
        'Std': f"{p.std():.0f}",
        'Hours ≥1000': f"{(p >= 1000).sum()}",
    })

print(f"{'Scenario':<15} {'Mean':>8} {'Min':>8} {'Max':>8} {'Std':>8} {'Hours≥1000':>12}")
print('-' * 65)
for s in price_stats:
    print(f"{s['Scenario']:<15} {s['Mean']:>8} {s['Min']:>8} {s['Max']:>8} {s['Std']:>8} {s['Hours ≥1000']:>12}")

---
## 步骤 3: 发电调度分析

各机组的调度量通过 Merit Order 决定:
1. 新能源 (风光) 优先调度 (边际成本最低)
2. 煤电作为基荷 (300 元/MWh)
3. 气电仅在高峰时段调用 (600 元/MWh)

In [ ]:
# 选择一个场景展示堆叠面积图
target = 'base'
if target not in scenarios:
    target = list(scenarios.keys())[0]

dispatch = scenarios[target]['dispatch']
pivot = dispatch.pivot_table(
    index='timestamp', columns='fuel_type', values='dispatch_mw', aggfunc='sum'
).fillna(0)

fuel_colors = {
    'coal': '#8B4513', 'gas': '#FF8C00', 'wind': '#2E8B57',
    'solar': '#FFD700', 'storage': '#9370DB',
}

fig2 = go.Figure()
for col in pivot.columns:
    fig2.add_trace(go.Scatter(
        x=pivot.index, y=pivot[col],
        mode='lines', name=col,
        stackgroup='one',
        line=dict(width=0.5, color=fuel_colors.get(col, '#999')),
    ))

fig2.update_layout(
    title=f'Generation Dispatch by Fuel Type — {target} Scenario',
    xaxis_title='Time',
    yaxis_title='Dispatch (MW)',
    hovermode='x unified',
    height=450,
)
fig2.show()

### 🔄 场景对比: 新能源消纳率

In [ ]:
renewable_share = []
for name, data in scenarios.items():
    d = data['dispatch']
    total = d['dispatch_mw'].sum()
    re = d[d['fuel_type'].isin(['wind', 'solar'])]['dispatch_mw'].sum()
    renewable_share.append({'Scenario': name, 'Renewable %': f'{re/total*100:.1f}%'})

print(f"{'Scenario':<15} {'Renewable Share':>18}")
print('-' * 35)
for r in renewable_share:
    print(f"{r['Scenario']:<15} {r['Renewable %']:>18}")

---
## 步骤 4: 智能体利润分析

三种智能体类型:
- **learning** (PPO): 强化学习智能体，动态报价
- **naive**: 按边际成本报价
- **strategic**: 策略性报价 (加价)

In [ ]:
fig3 = go.Figure()

for name, data in scenarios.items():
    agent_totals = data['profits'].groupby('agent')['profit_元'].sum().reset_index()
    fig3.add_trace(go.Bar(
        name=name,
        x=agent_totals['agent'],
        y=agent_totals['profit_元'],
        marker_color=colors.get(name, '#999'),
        text=agent_totals['profit_元'].apply(lambda x: f'{x:,.0f}'),
        textposition='outside',
    ))

fig3.update_layout(
    title='Agent Total Profit Comparison by Scenario',
    xaxis_title='Agent',
    yaxis_title='Total Profit (元)',
    barmode='group',
    height=450,
    legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01),
)
fig3.show()

### 📊 各智能体利润时序

In [ ]:
if target in scenarios:
    profits = scenarios[target]['profits']
    fig4 = go.Figure()
    
    for agent in profits['agent'].unique():
        agent_data = profits[profits['agent'] == agent]
        cumulative = agent_data['profit_元'].cumsum()
        fig4.add_trace(go.Scatter(
            x=agent_data['timestamp'],
            y=cumulative,
            mode='lines',
            name=agent,
            line=dict(width=2),
        ))
    
    fig4.update_layout(
        title=f'Cumulative Profit by Agent — {target} Scenario',
        xaxis_title='Time',
        yaxis_title='Cumulative Profit (元)',
        hovermode='x unified',
        height=450,
    )
    fig4.show()

---
## 步骤 5: 运行验证脚本

通过 Python 调用 `verify_simulation.py` 检查输出完整性。

In [ ]:
import subprocess
import json

for name in ['base', 'wind_high', 'summer_peak']:
    path = f'../outputs/{name}'
    try:
        result = subprocess.run(
            ['python', '../assume/verify_simulation.py', '--input', path],
            capture_output=True, text=True, timeout=30
        )
        summary = json.loads(result.stdout.split('{"passed"')[0].rsplit('}', 1)[0] + '}')
        status = '✓' if result.returncode == 0 else '✗'
        print(f"{status} {name}: passed={summary.get('passed', '?')} | "
              f"price={summary.get('stats', {}).get('avg_price_元_per_mwh', '?')} 元/MWh | "
              f"re={summary.get('stats', {}).get('renewable_share_pct', '?')}%")
    except Exception as e:
        print(f"✗ {name}: verify failed — {e}")

---
## 📝 学习笔记

- **Merit Order**: 发电机组按边际成本排序，最低者优先中标
- **Pay-as-Clear**: 所有中标机组按边际机组的报价结算 (单一价格)
- **Price Cap**: 中国省间现货报价上限 1500 元/MWh，限制极端价格
- **新能源优先**: 风光边际成本接近零，在 merit order 中永远排最前
- **场景对比价值**: 高风电场景降低平均出清价格; 夏季高峰推高价格

## 🤔 反思题

1. 为什么 wind_high 场景的平均价格低于 base 场景？
2. 夏季高峰时哪些机组新增了发电量？这些机组的边际成本有什么特点？
3. 为什么 strategic 智能体的利润高于 naive 智能体？
4. 如果去掉 price cap (1500 元/MWh)，极端供需紧张时价格会怎样变化？
5. 如何设计 reward function 让 learning agent 学会在高峰时段保留容量？